
# 01 从一笔交易理解因子与标签

> **本章要解决的问题**：你说“这个因子有效”，到底在说什么？因子、信号、标签、收益、成本，这五个词怎样在一笔交易里对齐？为什么很多回测一开始就错了？

## 本章导读

因子研究最容易犯的错，不是模型不够高级，而是**把“现在知道的”和“以后才知道的”混在一起**。这一章不急着上统计，先把一笔交易拆开：

1. 一笔交易的四个时刻；
2. 因子：把“此刻已知”变成可重复的分数；
3. 标签：未来兑现后用来批改的量；
4. 时间对齐：未来函数怎样混进来；
5. 对照：全做、筛过、被拒；
6. 成本：毛利、净利、乘数、价差；
7. 一个完整的小例子：RB 下破前的下行效率。

### 学习目标

- 给定一个信号时刻，能写出因子值和未来标签，并标注“当时可见/未来兑现”。
- 能识别三种泄漏：用未来数据、用当根收盘成交、标签重叠。
- 能对一个因子做“全做/筛过/被拒”三行对照。
- 能把点数成本换算成一手净利。
- 能填一张最小研究卡，并做一次未来扰动检查。

### 先修

第02章的期望/方差；会一点 pandas；不需要机器学习。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="01 从一笔交易理解因子与标签" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>01 从一笔交易理解因子与标签｜知识图</title>
<desc>本章主线：四个时刻对齐，因子用“当时可见”、标签用“未来兑现”，回测才不偷看。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a01" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao01" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag01" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">01</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">从一笔交易理解因子与标签</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：四个时刻对齐，因子用“当时可见”、标签用“未来兑现”，回测才不偷看。</text>
<text x="60" y="88" font-size="10" fill="#6E7B8B">① 当时可见的信息</text>
<text x="740" y="88" font-size="10" fill="#E07B39" text-anchor="end">② 未来才兑现的标签</text>
<path d="M 60 236 L 60 178 L 77.4 191.4 L 94.9 201.3 L 112.3 205.3 L 129.7 202.5 L 147.2 193.7 L 164.6 181.3 L 182.1 168.7 L 199.5 159.5 L 216.9 156.1 L 234.4 159.5 L 251.8 169.1 L 269.2 182.3 L 286.7 195.8 L 304.1 206.1 L 321.5 210.7 L 339 208.4 L 356.4 200 L 373.8 187.8 L 391.3 175.1 L 408.7 165.5 L 426.2 161.5 L 443.6 164.4 L 461 173.5 L 478.5 186.6 L 495.9 200.2 L 513.3 210.9 L 530.8 216 L 548.2 214.3 L 565.6 206.3 L 583.1 194.3 L 600.5 181.5 L 617.9 171.5 L 635.4 167 L 652.8 169.4 L 670.3 178 L 687.7 190.8 L 705.1 204.5 L 722.6 215.6 L 740 221.3 L 740 236" fill="#E6EFF7" stroke="#E6EFF7" stroke-width="0" stroke-linecap="round" stroke-linejoin="round"/>
<path d="M 60 178 L 77.4 191.4 L 94.9 201.3 L 112.3 205.3 L 129.7 202.5 L 147.2 193.7 L 164.6 181.3 L 182.1 168.7 L 199.5 159.5 L 216.9 156.1 L 234.4 159.5 L 251.8 169.1 L 269.2 182.3 L 286.7 195.8 L 304.1 206.1 L 321.5 210.7 L 339 208.4 L 356.4 200 L 373.8 187.8 L 391.3 175.1 L 408.7 165.5 L 426.2 161.5 L 443.6 164.4 L 461 173.5 L 478.5 186.6 L 495.9 200.2 L 513.3 210.9 L 530.8 216 L 548.2 214.3 L 565.6 206.3 L 583.1 194.3 L 600.5 181.5 L 617.9 171.5 L 635.4 167 L 652.8 169.4 L 670.3 178 L 687.7 190.8 L 705.1 204.5 L 722.6 215.6 L 740 221.3" fill="none" stroke="#1B3A5C" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
<line x1="130" y1="100" x2="130" y2="252" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-dasharray="5 4"/>
<circle cx="129.7" cy="202.5" r="4.2" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="1.6"/>
<line x1="310" y1="100" x2="310" y2="252" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-dasharray="5 4"/>
<circle cx="304.1" cy="206.1" r="4.2" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="1.6"/>
<line x1="490" y1="100" x2="490" y2="252" stroke="#E07B39" stroke-width="1.3" stroke-linecap="round" stroke-dasharray="5 4"/>
<circle cx="495.9" cy="200.2" r="4.2" fill="#E07B39" stroke="#E07B39" stroke-width="1.6"/>
<line x1="670" y1="100" x2="670" y2="252" stroke="#6E7B8B" stroke-width="1.3" stroke-linecap="round" stroke-dasharray="5 4"/>
<circle cx="670.3" cy="178" r="4.2" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="1.6"/>
<path d="M 60 262 L 745 262" fill="none" stroke="#1B3A5C" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a01)"/>
<circle cx="130" cy="262" r="6" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="2"/>
<circle cx="310" cy="262" r="6" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="2"/>
<circle cx="490" cy="262" r="6" fill="#E07B39" stroke="#E07B39" stroke-width="2"/>
<circle cx="670" cy="262" r="6" fill="#FFFFFF" stroke="#1B3A5C" stroke-width="2"/>
<rect x="54" y="280" width="152" height="98" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="130" y="321.2" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">T−2 特征截止</text>
<text x="130" y="340.7" font-size="10" fill="#6E7B8B" text-anchor="middle">09:59 收盘：最多看到这里</text>
<rect x="234" y="280" width="152" height="98" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="310" y="321.2" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">T−1 发出决定</text>
<text x="310" y="340.7" font-size="10" fill="#6E7B8B" text-anchor="middle">10:00:00：写下方向</text>
<rect x="414" y="280" width="152" height="98" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="490" y="321.2" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">T 最早成交</text>
<text x="490" y="340.7" font-size="10" fill="#6E7B8B" text-anchor="middle">10:00:01：只能按此价成交</text>
<rect x="594" y="280" width="152" height="98" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="670" y="321.2" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">T+1 标签结束</text>
<text x="670" y="340.7" font-size="10" fill="#6E7B8B" text-anchor="middle">10:10:00：未来兑现收益</text>
<path d="M 138 262 L 302 262" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag01)"/>
<path d="M 318 262 L 482 262" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag01)"/>
<path d="M 498 262 L 662 262" fill="none" stroke="#6E7B8B" stroke-width="1.2" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#ag01)"/>
<text x="60" y="398" font-size="10" fill="#6E7B8B">信息集边界：T−1 之后才成交，T+1 才有结果</text>
</svg>

> **图**：本章主线：四个时刻对齐，因子用“当时可见”、标签用“未来兑现”，回测才不偷看。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：因子研究员、实习生、策略开发。
- **输入**：分钟或日线行情、信号时刻、合约乘数。
- **输出**：因子值、未来标签、研究卡、三行对照表。
- **在流程的哪一步**：因子研究的第一步：定义“当时知道什么、未来兑现什么”。
- **失败模式**：未来函数、标签重叠、用中间价当成交价、只看毛利。
- **最小工作流**：定四个时刻 → 算因子 → 算标签 → 扰动未来检查 → 三行对照 → 扣成本。

---
## 1. 一笔交易的四个时刻

### 1.1 四个时刻

任何一笔可复现的交易，都要写清四个时刻：

| 时刻 | 含义 | 例子 |
|---|---|---|
| 特征截止 | 你最多能看到哪一刻的数据 | 09:59 收盘 |
| 发出决定 | 策略在何时决定开仓 | 10:00:00 |
| 最早成交 | 理论上最早能成交的时点 | 10:00:01 |
| 窗口结束 | 标签/退出对应的最后时刻 | 10:10:00 |

四个时刻错一个，回测就可能偷看未来。

### 1.2 一笔小账

做空信号在收盘 100 出现，下一时刻可卖价 99，退出可买价 98，乘数 10，每边手续费 3 元。

- 方向标签：$100-98=2$ 点；
- 实际可执行毛利：$(99-98)\times10=10$ 元；
- 净利：$10-2\times3=4$ 元。

**“方向对了 2 点”和“账户赚了 4 元”是两件事。** 研究排序可以用方向标签；评估交易必须用可执行价格和成本。

### 1.3 三个概念

- **特征**：此刻能算出来的数（动量、波动、成交量）；
- **因子**：把特征变成可重复分数的规则（含方向、窗口、缺失处理）；
- **标签**：未来兑现后用来批改的量（收益、是否延续、最大逆向）。

因子是“出题”，标签是“答案”，信号是“根据因子做的决定”。三者不能混。

### 1.4 因子挖掘不是“搜公式”

更准确的定义是：**提出可证伪的经济假设 → 构造数值信号 → 检验预测稳定性 → 中性化 → 组合 → 扣费/样本外**。公式只是实现，不是研究目的。

只从海量数据里搜公式，有三个风险：多重检验假阳性、没有经济逻辑（不知道何时失效）、过拟合。因子有 IC 不等于 Alpha：IC 是原材料，Alpha 是扣费、风险调整、样本外后仍成立的组合超额。

研究时要过四道门：**逻辑门**（经济机制）、**样本门**（新样本/极端市场）、**交易门**（换手/冲击成本）、**风险门**（风格/集中度暴露）。这四道门也是后面 11、17、37 的主线。

**常见错误**：

1. 用收盘价算信号，又用同一根收盘价成交。
2. 把“方向标签”当“账户收益”。
3. 不写乘数/手续费，只看点数。

**停下来想一想**：为什么“特征截止”和“最早成交”之间要留时间？

<details><summary>对照你的思路</summary>

因为从“看到数据”到“发出订单”再到“成交”需要时间。若特征截止是 10:00 收盘，你不可能在 10:00:00 用这个收盘价成交；最早也要 10:00:01 或下一根开盘。不留时间，就是未来函数。

</details>

---


### 2.2 一个例子：下行效率

突破前取 5 个收盘价 $p_0,\dots,p_4$，$p_0$ 最早、$p_4$ 最新：

$$E=\frac{p_0-p_4}{\sum_{i=1}^4|p_i-p_{i-1}|}.$$

**公式拆解**

- **符号**：$p_0$ 最早、$p_4$ 最新；分子是净跌幅，分母是 4 段绝对变化之和。
- **结构**：净位移 ÷ 实际路程；1=直线下跌，0=来回震荡，负=净上涨。
- **代入**：100、98、97、96、95 → 分子 $100-95=5$，分母 $2+1+1+1=5$，$E=1$。
- **边界**：5 个价全相等→分母 0→NaN；净上涨→$E<0$；分母 $\ge|$分子$|$，所以 $E\in[-1,1]$。
- **用途**：做因子排序/分组；不是概率，也不预测“继续跌”。

对照：100、98、100、97、95 终点同为 95，但中途回到 100，分母 $2+2+3+2=9$，$E=5/9\approx0.556$。同终点、不同路径，$E$ 不同。

**读公式四步**：符号 → 结构 → 代入 → 边界。以后遇到公式都按这四步。

### 2.3 因子 vs 信号


In [2]:
import numpy as np

def down_efficiency(prices):
    p = np.asarray(prices, dtype=float)
    travel = np.abs(np.diff(p)).sum()
    return (p[0] - p[-1]) / travel if travel > 0 else np.nan

paths = [
    [100, 98, 97, 96, 95],      # 直线下跌
    [100, 98, 100, 97, 95],     # 中间反弹
    [100, 98, 99, 97, 95],      # 小反弹
    [100, 100, 100, 100, 100],  # 横盘，分母为 0
]
for path in paths:
    print(path, "效率", round(float(down_efficiency(path)), 3))


[100, 98, 97, 96, 95] 效率 1.0
[100, 98, 100, 97, 95] 效率 0.556
[100, 98, 99, 97, 95] 效率 0.714
[100, 100, 100, 100, 100] 效率 nan


**读数与应用**：100→98→97→96→95 的效率是 1.0；同样从 100 到 95、但中途回到 100 的路径只有 0.556；小反弹是 0.714；横盘分母为 0，返回缺失，不能填成 0。

**常见错误**：

1. 把“当根 K 线”算进突破前的效率。当根还没结束，不能用。
2. 分母为 0 时填 0，把“没方向”误当成“弱方向”。
3. 分母只算下跌段、漏掉反弹段；这样“跌—反弹—跌”会被误判成直线下跌。
4. 因子方向写反。效率高是“跌得顺”，不是“跌得多”。

**停下来想一想**：效率高，是否一定意味着“接下来会继续跌”？

<details><summary>对照你的思路</summary>

不一定。因子只描述“突破前走得顺不顺”；是否延续，要用标签检验。效率高可能意味着“趋势强”，也可能意味着“已经跌太多、要反弹”。因子是假设，标签是检验。

</details>

---



## 3. 标签：未来兑现后用来批改的量

### 3.1 三种常见标签

| 标签 | 定义 | 回答的问题 |
|---|---|---|
| 固定期限收益 | $r_{t\to t+h}=P_{t+h}/P_t-1$ | 未来 $h$ 期涨跌多少 |
| MFE/MAE | 窗口内最大有利/不利变动 | 途中最好/最坏走到哪 |
| 是否先触线 | 先碰到止盈线还是止损线 | 路径先后顺序 |

同一个因子，用不同标签，结论可能不同。**标签决定研究问题。**

### 3.2 重叠标签

若标签是“未来 5 天收益”，相邻两天的样本共享 4 天行情：

| 信号日 | 标签窗口 |
|---|---|
| 周一 | 周二–下周一 |
| 周二 | 周三–下周二 |
| 周三 | 周四–下周三 |

这些标签高度相关，不能当独立样本；标准误会被低估，t 值会虚高。

### 3.3 代码：三种标签



In [3]:

import numpy as np

prices = np.array([100.0, 98, 97, 99, 96, 95])
entry = prices[0]
window = prices[1:]

print("5 期收益", round(window[-1] / entry - 1, 4))
print("MFE（最大有利，做空）", round(entry - window.min(), 2))
print("MAE（最大不利，做空）", round(entry - window.max(), 2))


5 期收益 -0.05
MFE（最大有利，做空） 5.0
MAE（最大不利，做空） 1.0



**读数与应用**：做空时，价格越低越有利，所以 MFE 用 entry 减最小值；MAE 用 entry 减最大值。方向反了，MFE/MAE 就反了。

**常见错误**：

1. 做空时把 MFE/MAE 的方向写反。
2. 用重叠标签当独立样本。
3. 只报固定期限收益，不看路径；一个策略可能期末赚、途中差点爆仓。

**停下来想一想**：为什么“先触线”标签更接近真实交易？

<details><summary>对照你的思路</summary>

真实交易有止损/止盈，路径先后决定你是否被扫出。固定期限收益只看终点，可能忽略“途中先触发止损”。先触线标签需要分钟/ tick 数据，且要规定“同根 K 线先碰哪边”的处理规则；处理不好会引入假设偏差。

</details>

---



## 4. 时间对齐：未来函数怎样混进来

### 4.1 三种泄漏

1. **特征泄漏**：特征用了 $t$ 之后的数据；
2. **标签泄漏**：标签窗口跨进测试段，或训练用了测试期标签；
3. **执行泄漏**：用当根收盘价成交，或假设订单在信号前已成交。

### 4.2 扰动未来检查

对特征做一次“未来扰动”：

1. 复制一份数据；
2. 把 $t$ 之后的价格改掉；
3. 重新算 $t$ 时刻的特征；
4. 若特征变了，说明它读了未来。

### 4.3 代码：正确 vs 泄漏



In [4]:

import numpy as np

def good_feature(prices):
    # 只用到 t=2 及之前
    return prices[2] / prices[0] - 1

def leaky_feature(prices):
    # 用了最后一根，未来
    return prices[-1] / prices[0] - 1

base = np.array([100.0, 101, 102, 103, 104])
perturbed = base.copy()
perturbed[3:] = [999, 999]

print("正确特征", round(good_feature(base), 4), round(good_feature(perturbed), 4))
print("泄漏特征", round(leaky_feature(base), 4), round(leaky_feature(perturbed), 4))


正确特征 0.02 0.02
泄漏特征 0.04 8.99



**读数与应用**：扰动未来后，正确特征不变；泄漏特征变了。这个检查比重跑回测更快，适合做因子准入。

**常见错误**：

1. 只扰动收盘价，不扰动成交量/持仓量/修订后的财务数据。
2. 特征正确，但标签用了未来；扰动特征查不出来。
3. 数据源本身已经倒填修订值；扰动行不一定发现。

**停下来想一想**：如果特征正确、标签也正确，还会不会有泄漏？

<details><summary>对照你的思路</summary>

会。执行泄漏：用当根收盘价成交、用信号前的盘口、假设订单在信号前已成交；样本泄漏：训练集和测试集共享同一段行情；选择泄漏：用测试集挑因子。泄漏不只发生在特征里。

</details>

---



## 5. 对照：全做、筛过、被拒

### 5.1 三行账

只看“筛过”的信号，会把市场本来就有的方向归功于因子。正确做法是同时记三行：

| 组 | 含义 | 要回答 |
|---|---|---|
| 全做 | 所有满足事件定义的信号 | 这类事件本来平均赚多少？ |
| 筛过 | 因子分数高、实际开仓的 | 筛子留下了什么？ |
| 被拒 | 分数低、没有开仓的 | 筛子错过了什么？ |

若“筛过”比“被拒”好，说明因子有区分力；若“被拒”反而更好，筛子可能在做反。

### 5.2 影子账

被拒的信号不能直接消失。按同一套入场/退出/成本规则，给它们记一笔“影子账”。这样才知道过滤器是“挑到了好机会”，还是“只是少做了交易”。

### 5.3 同期比较

不同日期的行情不同。比较高分组和低分组，最好在同一交易日内配对；否则“高分组恰好集中在上涨日”会污染结论。

### 5.4 代码：三行对照



In [5]:

import pandas as pd

df = pd.DataFrame({
    "group": ["全做", "筛过", "被拒"],
    "n": [100, 30, 70],
    "gross": [0.20, -0.10, 0.30],
    "cost": [0.50, 0.50, 0.50],
})
df["net"] = df.gross - df.cost
print(df)


  group    n  gross  cost  net
0    全做  100    0.2   0.5 -0.3
1    筛过   30   -0.1   0.5 -0.6
2    被拒   70    0.3   0.5 -0.2



**读数与应用**：全做净 -0.30，筛过净 -0.60，被拒净 -0.20。筛子不但没帮忙，还把更好的机会拒掉了。只看“筛过”的毛利，会以为“至少没亏太多”；加上被拒的影子账，才知道过滤器在破坏价值。

**常见错误**：

1. 只报筛过的收益，不报被拒的影子账。
2. 用不同日期/不同市场状态比较多空组。
3. 把“少做交易”当成“风险降低”。少做也可能错过好机会。

**停下来想一想**：筛过组亏钱，是否一定说明因子无效？

<details><summary>对照你的思路</summary>

不一定。可能成本太高、样本期太短、筛过组的行情特殊；也可能因子方向反了。要同时看：全做/筛过/被拒的差、同期配对、样本外、成本敏感性。单看一行，结论不可靠。

</details>

---



## 6. 成本：从点数到一手净利

### 6.1 四个成本

- 手续费：按手/按成交额；
- 买卖价差：入场买价、出场卖价；
- 滑点：下单到成交的价格差；
- 冲击成本：大单改变价格。

### 6.2 单位

- 报价单位：元/吨、点；
- 合约乘数：吨/手、点/手；
- 一手盈亏 = 点数 × 乘数；
- 净利 = 毛利 − 手续费 − 价差 − 滑点。

### 6.3 手算

做空 2 点，乘数 10，每边手续费 3 元，价差 1 点：

- 毛利：$2\times10=20$ 元；
- 手续费：$2\times3=6$ 元；
- 价差：$1\times10=10$ 元；
- 净利：$20-6-10=4$ 元。

**价差和手续费会吃掉大部分“看起来很小”的边际。**

### 6.4 代码：一手账单



In [6]:

points, mult, fee, spread_points = 2, 10, 6, 1

gross = points * mult
net = gross - fee - spread_points * mult
print("毛利", gross, "净利", net)
print("成本占毛利", round((gross - net) / gross, 3))


毛利 20 净利 4
成本占毛利 0.8



**读数与应用**：2 点的毛利，扣完只剩 4 元，成本占 80%。很多因子在毛收益上“看起来有效”，扣完成本就没了。

**常见错误**：

1. 只算手续费，不算价差/滑点。
2. 把“点”直接当“元”。乘数是最容易漏的。
3. 用单边成本当双边。开仓+平仓都要算。

**停下来想一想**：为什么高频策略对成本特别敏感？

<details><summary>对照你的思路</summary>

高频策略每笔毛利润很薄，交易次数多；手续费、价差、滑点按次累积，很快吃掉毛利。而且高频对执行延迟、盘口深度更敏感。成本模型错了，回测会从“赚”变成“亏”。

</details>

---



## 7. 一个完整的小例子

### 7.1 研究卡

- 问题：RB 下破前“跌得顺”，下破后更容易延续吗？
- 事件：5 分钟收盘跌破前两根最低价；
- 因子：突破前 5 个收盘价的下行效率；
- 标签：突破后 10 分钟的方向变动（点）；
- 成本：手续费 6 元/手，价差 1 点，乘数 10；
- 对照：全做 / 筛过（效率 > 0.6）/ 被拒。

### 7.2 五条事件

| 事件 | 效率 | 方向变动（点） | 毛利 | 净利 |
|---|---:|---:|---:|---:|
| E1 | 0.8 | +3 | 30 | 14 |
| E2 | 0.5 | −2 | −20 | −36 |
| E3 | 0.9 | +4 | 40 | 24 |
| E4 | 0.2 | −1 | −10 | −26 |
| E5 | 0.6 | +1 | 10 | −6 |

净利 = 点数 × 10 − 6 − 10。

### 7.3 读表

- 全做平均净利：$(14-36+24-26-6)/5=-6$ 元；
- 筛过（效率 > 0.6）：E1、E3，平均 $(14+24)/2=19$ 元；
- 被拒：E2、E4、E5，平均 $(-36-26-6)/3\approx-22.7$ 元。

筛过组明显更好，但只有 2 笔，样本太小；不能据此下结论。

### 7.4 代码：算这张表



In [7]:

import pandas as pd

df = pd.DataFrame({
    "event": ["E1", "E2", "E3", "E4", "E5"],
    "eff": [0.8, 0.5, 0.9, 0.2, 0.6],
    "move": [3, -2, 4, -1, 1],
})
df["gross"] = df["move"] * 10
df["net"] = df["gross"] - 6 - 1 * 10
df["selected"] = df["eff"] > 0.6

print(df)
print("全做平均净利", round(df["net"].mean(), 2))
print("筛过平均净利", round(df.loc[df["selected"], "net"].mean(), 2))
print("被拒平均净利", round(df.loc[~df["selected"], "net"].mean(), 2))


  event  eff  move  gross  net  selected
0    E1  0.8     3     30   14      True
1    E2  0.5    -2    -20  -36     False
2    E3  0.9     4     40   24      True
3    E4  0.2    -1    -10  -26     False
4    E5  0.6     1     10   -6     False
全做平均净利 -6.0
筛过平均净利 19.0
被拒平均净利 -22.67



**读数与应用**：筛过组平均 19 元，被拒组约 -22.7 元；差距很大，但筛过组只有 2 笔。**样本量、独立交易日、成本敏感性**都要看，不能只看均值。

### 7.5 研究卡检查清单

- [ ] 事件定义可复现（时间、价格、缺口处理）；
- [ ] 因子只用事件前已知数据；
- [ ] 标签窗口明确，重叠样本标记；
- [ ] 入场/出场价格可成交；
- [ ] 成本含手续费、价差、滑点；
- [ ] 全做/筛过/被拒三行都有；
- [ ] 样本量、独立交易日、样本外；
- [ ] 结论能被反例推翻。

**常见错误**：

1. 用全样本分位数切阈值，再回测同一段。
2. 筛过组只有几笔，就宣称“因子有效”。
3. 不报被拒组，过滤器的作用说不清。

**停下来想一想**：这个例子里，最大的不确定性是什么？

<details><summary>对照你的思路</summary>

样本太小：只有 5 笔、筛过组 2 笔；没有独立交易日、没有样本外。均值差距可能由一两笔极端值造成。要扩大样本、按日期分组、做成本敏感性，才能判断差距是否稳定。

</details>

---



## 本章小结

### 一张表

| 概念 | 要点 |
|---|---|
| 四个时刻 | 特征截止、决定、最早成交、窗口结束 |
| 因子 | 截至 $t$ 已知的数据 → 可重复分数 |
| 标签 | 未来兑现的批改量；固定期限/MFE/MAE/先触线 |
| 泄漏 | 特征、标签、执行、样本、选择 |
| 对照 | 全做/筛过/被拒 + 影子账 |
| 成本 | 手续费 + 价差 + 滑点；点数 × 乘数 |
| 研究卡 | 事件、因子、标签、执行、成本、对照 |

### 八句话

1. 方向标签不等于账户收益。
2. 因子是出题，标签是答案，信号是决定。
3. 标签决定研究问题；换标签可能换结论。
4. 重叠标签不能当独立样本。
5. 扰动未来是检查特征泄漏的最快方法。
6. 只看筛过组，会把市场方向归功于因子。
7. 点数 × 乘数才是钱；成本按双边算。
8. 小样本的“显著”最不可信。

---

## 本章习题

**1.** 信号在 10:00 收盘出现，10:00:01 最早成交，10:10 窗口结束。写出四个时刻。

**2.** 价格 100、99、98、97、96。求下行效率。

**3.** 做空入场 100，窗口内最高 102、最低 95。求 MFE、MAE。

**4.** 标签是未来 5 天收益，相邻两天样本为什么不能当独立？

**5.** 做空 3 点，乘数 10，每边手续费 4 元，价差 1 点。求净利。

**6.** 全做净利 0.2，筛过 0.5，被拒 0.1。筛子有没有区分力？

**7.** 特征在扰动未来后变了，说明什么？怎么修？

**8.** 写出一张最小研究卡的 6 个要素。

<details><summary>参考答案</summary>

**1.** 特征截止 10:00 收盘；决定 10:00:00；最早成交 10:00:01；窗口结束 10:10:00。

**2.** 净跌 4，路程 4，效率 1.0。

**3.** 做空 MFE $=100-95=5$；MAE $=100-102=-2$（即最大不利 2 点）。

**4.** 相邻两天的 5 日标签共享 4 天行情，高度相关；标准误被低估，t 值虚高。

**5.** 毛利 30；手续费 8；价差 10；净利 12 元。

**6.** 筛过 0.5 > 被拒 0.1，有区分力；但还要看样本量、同期、成本、样本外。

**7.** 特征读了未来；改成只用 $t$ 及之前的数据，重做扰动检查。

**8.** 事件定义、因子公式、标签定义、执行价格、成本、对照/样本外。

</details>

---

## 参考资料

- Alphalens 文档（因子报告：https://github.com/quantopian/alphalens）。
- Grinold, R. C. & Kahn, R. N. *Active Portfolio Management*（因子/组合）。
- López de Prado, M. *Advances in Financial Machine Learning*（标签、泄漏、验证）。
